# Museum Heist -- PG-2

This is the **Reinforcement Learning** project developed during Erasmus exchange at the University of Milan.
The project focuses on stochastic policies and policy gradient, and it's scientific objective
is to investigate the challenges related to strategic environments that require the use of stochastic policies.

The information below is just a small debrief of the task.
For more detailed task description please refer to the **README.md** file in this repository.

## Problem Description

A thief is trying to steal a specific painting from an art museum.

A security camera is located in the correspondence of each exhibition room,
but the security guard in the control room can only watch one video feed at a time.
Once per minute, an algorithm selects the next location to show on the screen.

Simultaneously, the thief can move to a nearby room or stay in the same room.
The thief starts from an unknown initial room, has to reach an unknown target room
(the one with the painting they want to steal) and go back to the initial room in order to escape.
The thief has a map of the museum and is also in radio contact with a hacker who knows the current active security camera.

The episode ends if:
- the surveillance algorithm selects the room currently occupied by the thief,
- or if it selects the target room after the painting has been stolen, but the thief has already escaped.

In [24]:
# imports
import gymnasium as gym
from gymnasium import spaces
import networkx as nx
import numpy as np

from utils import map_museum_to_graph

## Environment and Thief

what we have, how it's modelled, motivate choice wrt state/action structure and difficulty

In [25]:
# Model the environment in a way, that graph vertices are rooms and edges are doors between rooms.
# We want to design environment in a grid-way style, that is:
# -- graph vertices are organized into n x m grid
# -- each vertex is connected to at most 4 other vertices
# networkx grid_2d_graph makes all the work for us.

# we pass into env function the map of the museum, so that we can easily create other museum topologies
# with walls, etc.

# Even though the security camera (agent) is stateless, the environment itself needs to track everything to run the simulation and provide rewards.
class MuseumHeistEnv(gym.Env):
    """
    Museum Heist Environment
    Agent: Security Camera (stateless)
    Adversary: Thief (stateful, shortest-path logic)
    """
    def __init__(self, museum_map, beta=2.0):
        # Initialize the learning environment.
        super().__init__()

        # Environmental metadata.
        self.beta = beta
        self.museum_map = museum_map
        self.height = len(museum_map)
        self.width = len(museum_map[0])

        # Museum 2D grid topology, where graph nodes are denoted by (x, y).
        # We map each graph node (x, y) to integer from 0 to N-1 and vice versa to represent room numbers.
        self.museum_graph, self.num_rooms = map_museum_to_graph(museum_map)
        self.node_to_idx = {n: i for i, n in enumerate(self.museum_graph.nodes())}
        self.idx_to_node = {i: n for n, i in self.node_to_idx.items()}

        # Gym spaces -- Action (room selection) & Observation space for an agent,
        # since policy is stochastic, the observation always returns 0 (stateless).
        self.action_space = spaces.Discrete(self.num_rooms)
        self.observation_space = spaces.Discrete(1)

        # Internal state variables, for cost calculation and episode finish.
        # n_counts represents for each room, the number of rounds since i-th room was last selected by surveillance algorithm.
        # Cost to enter room i-th is 1 + self.beta * 2 ^ { -(n_counts[i] - 1) }
        self.n_counts = np.ones(self.num_rooms)
        self.thief_pos = None
        self.painting_pos = None
        self.start_pos = None
        self.has_painting = False

    def reset(self, seed=None, options=None):
        # Restart the environment after episode finish.
        super().reset(seed=seed)

        # Choose Thief starting position and painting position at random at the beginning of each episode.
        # We choose two at once with replace=False to avoid duplicating the same position.
        positions = self.np_random.choice(self.num_rooms, 2, replace=False)
        self.start_pos = positions[0]
        self.painting_pos = positions[1]

        # Initialize episode initial data at each reset.
        self.thief_pos = self.start_pos
        self.has_painting = False
        self.n_counts = np.ones(self.num_rooms)

        # Return nothing, as we're stateless.
        return 0, {}

    def _calculate_next_thief_step(self):
        # Perform one step of Thief in the museum topology.
        # Since Thief next move is state/environmental-dependant, based on dynamic cost,
        # we calculate the shortest path inside it and return next step on Thief's path.

        # Our museum graph has to be a DIRECTED graph, since the cost of entering i-th room
        # from j-th room can be different from entering j-th room from i-th room, since n_counts[i] != n_counts[j].
        # m_graph is the graph with current state/costs of entering each room.
        # For every edge in the initial Museum Map, we calculate both edges of entering u and v.
        m_graph = nx.DiGraph()
        for u, v in self.museum_graph.edges():
            u_idx, v_idx = self.node_to_idx[u], self.node_to_idx[v]

            # First direction
            cost_v = 1.0 + self.beta * (2.0 ** -(self.n_counts[v_idx] - 1))
            m_graph.add_edge(u_idx, v_idx, weight=cost_v)

            # Second direction
            cost_u = 1.0 + self.beta * (2.0 ** -(self.n_counts[u_idx] - 1))
            m_graph.add_edge(v_idx, u_idx, weight=cost_u)

        current_target_room = self.start_pos if self.has_painting else self.painting_pos

        try:
            path = nx.shortest_path(m_graph, source=self.thief_pos, target=current_target_room, weight='weight')
            if len(path) > 1:   # If the Thief is not standing in the node that source == target.
                return path[1]  # The next step on the shortest path, path[0] is the source.
            return self.thief_pos
        except nx.NetworkXNoPath:
            return self.thief_pos

    def step(self, action):
        # Perform one step of the environment (Thief movement, surveillance action).
        # Return is (observation -- 0 since stateless, reward, terminate, truncated, info)

        # 1. Update surveillance counters
        self.n_counts += 1
        self.n_counts[action] = 1

        # 2. Check whether Agent caught the Thief, reward if so.
        # Chosen room is the action fed into function (action taken by the agent).
        if action == self.thief_pos:
            return 0, 10.0, True, False, {"reason": "thief caught"}

        # 3. If Thief was not caught, move him to next position with already updated n_counts.
        self.thief_pos = self._calculate_next_thief_step()

        # 4. Verify whether Thief still has the painting to steal.
        if not self.has_painting and self.thief_pos == self.painting_pos:
            self.has_painting = True

        # 5. Check whether Agent let the Thief escape with the painting, reward accordingly.
        if self.has_painting and self.thief_pos == self.start_pos:
            return 0, -10.0, True, False, {"reason": "thief escaped"}

        # Episode continues with small time penalty
        return 0, -0.1, False, False, {}

In [27]:
custom_map = [
    "OOOOO",
    "OOOOO",
    "OOOOO",
    "OOOOO",
    "OOOOO",
]

env = MuseumHeistEnv(museum_map=custom_map, beta=2.0)

# Agent -- The Security Camera (Stateless Softmax Policy Gradient)

some descrption